# 🌍 Earthquake Prediction FastAPI + Ngrok + LSTM
This notebook sets up a FastAPI server to handle earthquake waveform prediction using your LSTM model and exposes it via ngrok for public access.

In [ ]:
# ✅ Step 1: Install dependencies
!pip install fastapi uvicorn pyngrok obspy keras tensorflow nest-asyncio

In [ ]:
# ✅ Step 2: Setup ngrok tunnel
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

# Kill previous tunnels if any
ngrok.kill()
public_url = ngrok.connect(8000)
print("🚀 Public URL:", public_url)

In [ ]:
# ✅ Step 3: Create FastAPI app in a separate file
%%writefile earthquake_api.py
from fastapi import FastAPI, UploadFile, File
import numpy as np
from keras.models import load_model
from obspy import read
from obspy.signal.filter import bandpass

app = FastAPI()
model = load_model("earthquake_lstm_model.h5")
PREDICTION_THRESHOLD = 0.7

@app.get("/")
def root():
    return {"message": "Earthquake Prediction API is running"}

@app.post("/predict/")
async def predict_earthquake(file: UploadFile = File(...)):
    try:
        st = read(file.file)
        tr = st[0]
        data = tr.data
        data = bandpass(data, freqmin=0.5, freqmax=10, df=tr.stats.sampling_rate, corners=2, zerophase=True)
        data = (data - np.mean(data)) / np.std(data)
        data = data.reshape(1, -1, 1)
        prediction = model.predict(data)[0][0]
        result = "yes" if prediction >= PREDICTION_THRESHOLD else "no"
        return {"result": result, "confidence": float(prediction)}
    except Exception as e:
        return {"error": str(e)}

In [ ]:
# ✅ Step 4: Run the API server in the background
!nohup uvicorn earthquake_api:app --host 0.0.0.0 --port 8000 &

In [ ]:
# ✅ Step 5: Test the API (replace URL and file path)
import requests
url = "http://YOUR_NGROK_URL_HERE/predict/"
files = {'file': open("YOUR_FILE_PATH.mseed", 'rb')}
response = requests.post(url, files=files)
print("Status Code:", response.status_code)
try:
    print("Response JSON:", response.json())
except Exception as e:
    print("Error reading JSON:", e, response.text)